In [ ]:
import os
import shutil
import pycolmap
from PIL import Image

image_path = './data/lego/test'
depth_path = './data/lego/depths'

depth_files = os.listdir(depth_path)
image_files = os.listdir(image_path)

#for i in range(len(image_files)):
#    img_path = f'{image_path}/{image_files[i]}'
#    dep_path = f'{depth_path}/{depth_files[i]}'
#    img = PIL.Image.open(img_path).convert('RGBA')
#    d_img = PIL.Image.open(dep_path).convert('RGBA')
#
#    filename = os.path.basename(image_files[i])
#
#    bg_w = PIL.Image.new("RGBA", img.size, "WHITE")
#    bg_b = PIL.Image.new("RGBA", d_img.size, "BLACK")
#
#    bg_w.paste(img, (0, 0), img)
#    bg_b.paste(d_img, (0, 0), d_img)
#
#    final_image = bg_w.convert("RGB")
#    final_image.save(img_path)
#    final_image_b = bg_b.convert("RGB")
#    final_image_b.save(dep_path)

def undistort_files(base_path: str, images_path: str):
    pycolmap.undistort_images(
        output_path=f"./{base_path}/undistorted/", 
        input_path=f'{base_path}/sparse/0', 
        image_path=f'{base_path}/{images_path}'
    )

    source = f'{base_path}/undistorted/sparse'
    dest = f'{base_path}/undistorted/sparse/0'

    if not os.path.exists(dest):
        os.makedirs(dest)

    for item_name in os.listdir(source):
        source_item = os.path.join(source, item_name)

        if item_name == '0':
            continue

        shutil.move(source_item, dest)

    paths = os.listdir(f'./{base_path}/undistorted/images')
    os.makedirs(f'./{base_path}/undistorted/depths', exist_ok=True)
    for i in range(len(paths)):
        filename = os.path.basename(paths[i])

        img = Image.open(f'./{base_path}/undistorted/images/{filename}')
        d_img = Image.open(f'./{base_path}/depths/{filename}')

        resized_img = d_img.resize(img.size, Image.Resampling.LANCZOS)
        resized_img.save(f'./{base_path}/undistorted/depths/{filename}')

undistort_files('data/lego', 'test')

In [6]:
from PIL import Image

image_path = './output/ficus_whitebg-trained/images'
depth_path = './output/ficus_whitebg-trained/depths'

depth_files = os.listdir(depth_path)
image_files = os.listdir(image_path)
for i in range(len(image_files)):
    img_path = f'{image_path}/{image_files[i]}'
    dep_path = f'{depth_path}/{depth_files[i]}'
    img = Image.open(img_path).convert('RGBA')
    d_img = Image.open(dep_path).convert('RGBA')

    filename = os.path.basename(image_files[i])

    bg_w = Image.new("RGBA", img.size, "WHITE")
    bg_b = Image.new("RGBA", d_img.size, "BLACK")

    bg_w.paste(img, (0, 0), img)
    bg_b.paste(d_img, (0, 0), d_img)

    final_image = bg_w.convert("RGB")
    final_image.save(img_path)
    final_image_b = bg_b.convert("RGB")
    final_image_b.save(dep_path)

# Training

In [ ]:
from gaussian_splatting.train import prepare_training, training

dataset, gaussians, trainer = prepare_training(sh_degree=3, source="./data/lego/undistorted", device='cuda', mode='densify', load_depth=True, with_scale_reg=True)
training(dataset=dataset, gaussians=gaussians, trainer=trainer, destination='./output/lego', iteration=15000, save_iterations=[7000, 15000], empty_cache_every_step=True)

# Physics

In [44]:
import os
import torchvision
from gaussian_splatting import GaussianModel, Camera
import torch


def render(gaussians: GaussianModel, camera: Camera, camera_id: int, pyhs_iter: int, save: str):
    os.makedirs(save, exist_ok=True)
    out = gaussians(camera)
    rendering = out['render']
    torchvision.utils.save_image(rendering, os.path.join(save, f'{camera_id}' + '_{0:05d}'.format(pyhs_iter) + ".png"))

### Against the wall, simple gravity mechanic applied equally to all gaussians but clamped to -5, 5.

In [ ]:
from gaussian_splatting.dataset import JSONCameraDataset

# variables to set
dt = .02
camera_id = 0
model_results = 'output/ficus_whitebg-trained/results'
gravity = torch.tensor([0., -1., 0.], device=gaussians.get_xyz.device)

# setup the gaussians
gaussians = GaussianModel(3).to('cuda')

# grab the camera data from the pretrained json file
cameras = JSONCameraDataset('output/ficus_whitebg-trained/cameras.json').to('cuda')

# pick a pretrained set, this one can be 7000, 30,000, or 60,000
gaussians.load_ply('output/ficus_whitebg-trained/point_cloud/iteration_30000/point_cloud.ply')

# alright, lets go!
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        velocity += gravity * dt
        points = torch.clamp(gaussians.get_xyz + velocity * dt, min=-5., max=5.)
        gaussians.get_xyz.data.copy_(points)

        frame = render(gaussians, camera, camera_id, step, model_results)

#### Output 1 video

There will be others, but in order to generate this video, be sure to install [ffmpeg](https://www.ffmpeg.org/).

In [48]:
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 wall_smash.mp4 -y

1278.69s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

### Fall Down, but with Mass

In [ ]:
# variables to set
dt = .02
camera_id = 0
model_results = 'output/ficus_whitebg-trained/results'
gravity = torch.tensor([0., 0., -1.], device=gaussians.get_xyz.device)

# setup the gaussians
gaussians = GaussianModel(3).to('cuda')

# grab the camera data from the pretrained json file
cameras = JSONCameraDataset('output/ficus_whitebg-trained/cameras.json').to('cuda')

# pick a pretrained set, this one can be 7000, 30,000, or 60,000
gaussians.load_ply('output/ficus_whitebg-trained/point_cloud/iteration_30000/point_cloud.ply')

# alright, lets go!dt = .02
camera_id = 0
model_results = 'output/ficus_whitebg-trained/results'

n = gaussians.get_xyz.shape[0]

indices = torch.randperm(n, device=gaussians.get_xyz.device)

camera = cameras[camera_id]
inv_mass = 1 / torch.rand((n, 1), device=gaussians.get_xyz.device)

velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        velocity += gravity * dt
        points = torch.clamp(gaussians.get_xyz + velocity * inv_mass * dt, min=-10., max=10.)
        gaussians.get_xyz.data.copy_(points)

        frame = render(gaussians, camera, camera_id, step, model_results)

TypeError: rand() received an invalid combination of arguments - got (tuple, device=torch.device), but expected one of:
 * (tuple of ints size, *, torch.Generator generator, tuple of names names, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, torch.Generator generator, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, tuple of names names, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [ ]:
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 mass_falling.mp4 -y

745.42s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena